# From one protected attribute to intersectional reweighing

This notebook compares four approaches on the public Adult Census Income dataset:

1. an unweighted baseline;
2. classic reweighing for one attribute;
3. multiplication of weights calculated separately;
4. joint tensor reweighing across `sex × age_group × zip_proxy × outcome`.

`zip_proxy` is **not a real ZIP code**. Adult contains race but no individual geography, so we create a reproducible, imperfect geographic proxy correlated with race. It lets us test whether mitigating a proxy also reduces disparities in the underlying attribute. Never present it as observed geography.

The weights are model-agnostic. Scikit-learn is used for training, Fairlearn for evaluation, and HolisticAI is included only as an optional one-attribute comparison.

## Fairness objectives

Demographic parity (equality of predicted outcomes) requires

$$P(\hat Y=1\mid A=a)=P(\hat Y=1\mid A=b).$$

Equality of opportunity requires equal true-positive rates:

$$P(\hat Y=1\mid Y=1,A=a)=P(\hat Y=1\mid Y=1,A=b).$$

Classic reweighing changes the training distribution so that the protected attribute and observed outcome are independent. It promotes demographic parity, but does not guarantee demographic parity or equality of opportunity in model predictions.

In [ ]:
# Run once if needed:
# %pip install -q pandas numpy scikit-learn fairlearn matplotlib seaborn xgboost holisticai

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, confusion_matrix
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load an independent public dataset

The data come from OpenML through scikit-learn, not from HolisticAI. HolisticAI's `adult` loader packages essentially the same benchmark for convenience.

In [ ]:
adult = fetch_openml(
    "adult", version=2, as_frame=True, parser="auto", data_home="./data"
)
raw = adult.frame.copy()

# Normalize the label used by different OpenML versions.
target_col = adult.target.name
raw["target"] = raw[target_col].astype(str).str.replace(".", "", regex=False).eq(">50K").astype(int)
if target_col != "target":
    raw = raw.drop(columns=[target_col])

# Convert OpenML missing markers.
raw = raw.replace("?", np.nan)
print(raw.shape)
raw[["age", "sex", "race", "target"]].head()

## 2. Construct protected groups and a simulated ZIP proxy

We retain real `race` only for evaluation. The simulated ZIP proxy is generated conditionally on race: most individuals are assigned to one of two ZIP-like areas associated with their race, while the remainder are assigned elsewhere. This mimics an imperfect geographic proxy without claiming that Adult contains real location data.

In [ ]:
def make_zip_proxy(race, seed=42, n_zips=12, concentration=0.80):
    rng = np.random.default_rng(seed)
    race = pd.Series(race).fillna("Unknown").astype(str)
    race_codes, race_levels = pd.factorize(race, sort=True)
    zips = np.array([f"ZCTA_{i:02d}" for i in range(n_zips)])
    out = []
    for rc in race_codes:
        # Each race has two preferred areas; assignments remain noisy.
        preferred = np.array([(2 * rc) % n_zips, (2 * rc + 1) % n_zips])
        if rng.random() < concentration:
            zi = rng.choice(preferred)
        else:
            zi = rng.integers(0, n_zips)
        out.append(zips[zi])
    return pd.Series(out, index=race.index, name="zip_proxy")


raw["age_group"] = pd.cut(
    pd.to_numeric(raw["age"], errors="coerce"),
    bins=[0, 39, 59, np.inf],
    labels=["under_40", "40_to_59", "60_plus"],
    include_lowest=True,
).astype("object").fillna("unknown")

raw["zip_proxy"] = make_zip_proxy(raw["race"], seed=RANDOM_STATE)

proxy_table = pd.crosstab(raw["zip_proxy"], raw["race"], normalize="index")
proxy_table.round(3)

In [ ]:
plt.figure(figsize=(11, 5))
sns.heatmap(proxy_table, cmap="Blues", annot=True, fmt=".2f")
plt.title("Race composition of the simulated ZIP proxy")
plt.xlabel("Observed race (evaluation only)")
plt.ylabel("Simulated ZIP proxy")
plt.tight_layout()

## 3. Split before calculating weights

Weights must be learned from the training data only. The test set remains unweighted and represents the population on which fairness and utility are evaluated.

In [ ]:
train_df, test_df = train_test_split(
    raw,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=raw["target"],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

sensitive_cols = ["sex", "race", "age_group", "zip_proxy"]

# Exclude direct sensitive variables and the simulated proxy from model features.
# They remain available for weighting and evaluation.
excluded = set(sensitive_cols + ["target", "age"])
feature_cols = [c for c in raw.columns if c not in excluded]

X_train = train_df[feature_cols]
y_train = train_df["target"].to_numpy()
X_test = test_df[feature_cols]
y_test = test_df["target"].to_numpy()

print("Train:", X_train.shape, "Test:", X_test.shape)

## 4. Model-agnostic reweighing functions

For any categorical grouping $A$ and binary outcome $Y$, classic reweighing uses

$$w(a,y)=\frac{P(A=a)P(Y=y)}{P(A=a,Y=y)}.$$

Passing several columns treats their Cartesian product as one intersectional group. No estimator appears in this calculation.

In [ ]:
def classic_reweighing(frame, group_cols, target_col="target", smoothing=0.0):
    # Return one weight per row for arbitrary categorical group intersections.
    cols = list(group_cols) + [target_col]
    data = frame[cols].copy()
    for c in group_cols:
        data[c] = data[c].astype("object").fillna("unknown")

    n = len(data)
    group_n = data.groupby(list(group_cols), observed=False).size().rename("n_group")
    target_n = data.groupby(target_col, observed=False).size().rename("n_target")
    joint_n = data.groupby(cols, observed=False).size().rename("n_joint")

    table = joint_n.reset_index()
    table = table.merge(group_n.reset_index(), on=list(group_cols))
    table = table.merge(target_n.reset_index(), on=target_col)

    # Optional additive smoothing protects sparse cells in larger tensors.
    table["weight"] = (
        table["n_group"] * table["n_target"]
        / (n * (table["n_joint"] + smoothing))
    )
    out = data.reset_index().merge(table[cols + ["weight"]], on=cols, how="left")
    return out.sort_values("index")["weight"].to_numpy()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    return w / np.mean(w)


def clip_and_normalize(w, lower=0.1, upper=10.0):
    return normalize_weights(np.clip(w, lower, upper))


def effective_sample_size(w):
    w = np.asarray(w, dtype=float)
    return w.sum() ** 2 / np.square(w).sum()

## 5. Four weighting strategies

`product_marginal` multiplies separately estimated corrections. `joint_tensor` estimates the full intersection directly. They are not mathematically equivalent.

In [ ]:
w_none = np.ones(len(train_df))
w_sex = classic_reweighing(train_df, ["sex"])
w_zip = classic_reweighing(train_df, ["zip_proxy"])

w_age = classic_reweighing(train_df, ["age_group"])
w_product = normalize_weights(w_sex * w_age * w_zip)

w_joint = classic_reweighing(
    train_df,
    ["sex", "age_group", "zip_proxy"],
)

# Keep both raw and stabilized joint weights to expose the sparse-cell trade-off.
w_joint_stable = clip_and_normalize(w_joint, lower=0.1, upper=10.0)

weights = {
    "baseline": w_none,
    "single_sex": normalize_weights(w_sex),
    "single_zip_proxy": normalize_weights(w_zip),
    "product_marginal": w_product,
    "joint_tensor": normalize_weights(w_joint),
    "joint_tensor_clipped": w_joint_stable,
}

pd.DataFrame({
    name: {
        "min": np.min(w),
        "median": np.median(w),
        "max": np.max(w),
        "ESS": effective_sample_size(w),
        "ESS_pct": effective_sample_size(w) / len(w),
    }
    for name, w in weights.items()
}).T.round(3)

## 6. Equivalent tensor calculation with Einstein summation

This section verifies that the groupby formulation and a tensor formulation produce the same joint weights. `einsum` performs marginalization; it does not define fairness by itself.

In [ ]:
def tensor_reweighing(frame, group_cols, target_col="target"):
    encoded = []
    levels = []
    for col in group_cols:
        codes, cats = pd.factorize(frame[col].astype("object").fillna("unknown"), sort=True)
        encoded.append(codes)
        levels.append(cats)
    y_codes, y_levels = pd.factorize(frame[target_col], sort=True)

    shape = tuple(len(x) for x in levels) + (len(y_levels),)
    counts = np.zeros(shape, dtype=float)
    np.add.at(counts, tuple(encoded) + (y_codes,), 1.0)

    # For dimensions sex × age × zip × y.
    if len(group_cols) != 3:
        raise ValueError("This demonstrator expects exactly three sensitive dimensions.")
    group_counts = np.einsum("sazy->saz", counts)
    target_counts = np.einsum("sazy->y", counts)
    total = np.einsum("sazy->", counts)
    expected = np.einsum("saz,y->sazy", group_counts / total, target_counts / total)
    observed = counts / total
    weight_tensor = np.divide(
        expected,
        observed,
        out=np.zeros_like(expected),
        where=observed > 0,
    )
    row_weights = weight_tensor[tuple(encoded) + (y_codes,)]
    return row_weights, counts, weight_tensor


w_einsum, counts, weight_tensor = tensor_reweighing(
    train_df,
    ["sex", "age_group", "zip_proxy"],
)
print("Maximum difference:", np.max(np.abs(w_joint - w_einsum)))
assert np.allclose(w_joint, w_einsum)

## 7. Train the same model with each weight vector

The estimator receives only `sample_weight`. The fairness logic remains outside the model.

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_cols),
])


def fit_model(sample_weight):
    model = Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000, solver="liblinear")),
    ])
    model.fit(X_train, y_train, model__sample_weight=sample_weight)
    return model


models = {name: fit_model(w) for name, w in weights.items()}
predictions = {name: model.predict(X_test) for name, model in models.items()}
probabilities = {name: model.predict_proba(X_test)[:, 1] for name, model in models.items()}

## 8. Evaluate overall, marginal and intersectional fairness

All metrics are calculated on the original, unweighted test set. For multi-category attributes we report the range between the best and worst group. Signs can be added later when a privileged reference group is substantively justified.

In [ ]:
def group_gap(y_true, y_pred, groups, metric):
    mf = MetricFrame(metrics=metric, y_true=y_true, y_pred=y_pred, sensitive_features=groups)
    values = pd.Series(mf.by_group).dropna().astype(float)
    return values.max() - values.min(), values


def evaluate_model(name, y_pred, y_score):
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
    }
    attributes = {
        "sex": test_df["sex"],
        "age": test_df["age_group"],
        "zip": test_df["zip_proxy"],
        "race": test_df["race"],
        "intersection": test_df[["sex", "age_group", "zip_proxy"]],
    }
    for attr_name, groups in attributes.items():
        dp, _ = group_gap(y_test, y_pred, groups, selection_rate)
        eo, _ = group_gap(y_test, y_pred, groups, recall_score)
        fpr, _ = group_gap(y_test, y_pred, groups, false_positive_rate)
        row[f"DP_gap_{attr_name}"] = dp
        row[f"EO_gap_{attr_name}"] = eo
        row[f"FPR_gap_{attr_name}"] = fpr
    return row


results = pd.DataFrame([
    evaluate_model(name, predictions[name], probabilities[name])
    for name in predictions
]).set_index("model")

results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
results["accuracy"].sort_values().plot.barh(ax=axes[0], title="Accuracy")
results["DP_gap_intersection"].sort_values().plot.barh(ax=axes[1], title="Intersectional DP gap")
results["EO_gap_intersection"].sort_values().plot.barh(ax=axes[2], title="Intersectional EO gap")
for ax in axes:
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()

## 9. Did correcting the ZIP proxy help race?

This is the key proxy experiment. Compare `single_zip_proxy` and `joint_tensor` against the baseline using both ZIP-based and race-based gaps. Improvement in one does not guarantee improvement in the other.

In [ ]:
proxy_columns = [
    "accuracy",
    "DP_gap_zip", "EO_gap_zip", "FPR_gap_zip",
    "DP_gap_race", "EO_gap_race", "FPR_gap_race",
]
results.loc[["baseline", "single_zip_proxy", "product_marginal", "joint_tensor", "joint_tensor_clipped"], proxy_columns].round(4)

## 10. Optional comparison with HolisticAI

HolisticAI's standard `Reweighing` API operates on `group_a` and `group_b`, so it naturally demonstrates one binary comparison at a time. The manual function above generalizes the same probability-ratio idea to multiple categories and intersections.

Library APIs can change between releases; run this cell only after installing the version used in the HolisticAI documentation.

In [ ]:
# Optional reference implementation for one binary attribute.
# from holisticai.bias.mitigation import Reweighing
#
# male = train_df["sex"].astype(str).eq("Male").to_numpy()
# female = train_df["sex"].astype(str).eq("Female").to_numpy()
# mitigator = Reweighing()
# mitigator.fit(y_train, group_a=male, group_b=female)
#
# HolisticAI can also be used through holisticai.pipeline.Pipeline as shown in:
# https://holisticai.readthedocs.io/en/latest/gallery/tutorials/bias/mitigating_bias/binary_classification/examples/example_census_data.html

## 11. Optional XGBoost `DMatrix`

`DMatrix` is not another fairness technique. It is only XGBoost's optimized container for passing the same model-agnostic weights to training.

In [ ]:
# Optional:
# import xgboost as xgb
# Xtr = models["baseline"].named_steps["prep"].transform(X_train)
# Xte = models["baseline"].named_steps["prep"].transform(X_test)
# dtrain = xgb.DMatrix(Xtr, label=y_train, weight=weights["joint_tensor_clipped"])
# dtest = xgb.DMatrix(Xte, label=y_test)
# booster = xgb.train(
#     {"objective": "binary:logistic", "eval_metric": "auc", "seed": RANDOM_STATE},
#     dtrain,
#     num_boost_round=200,
# )
# xgb_score = booster.predict(dtest)
# xgb_pred = (xgb_score >= 0.5).astype(int)

## Interpretation checklist

- Did the weighted training distribution become independent of the selected groups?
- Did demographic parity improve in predictions?
- Did equality of opportunity improve as well, or move in the opposite direction?
- Did marginal fairness hide a poor intersectional group?
- Did mitigating the ZIP proxy improve fairness measured on observed race?
- Did multiplication create more extreme weights than joint estimation?
- How much effective sample size was lost?
- Are conclusions stable under clipping, alternative age bins and another train/test split?

The correct conclusion is empirical: reweighing changes the influence of training observations, but it does not guarantee that every downstream fairness definition improves.

## Sources

- Kamiran, F. and Calders, T. *Data Preprocessing Techniques for Classification without Discrimination* (2012).
- [HolisticAI: Fair Classification in US Census Data](https://holisticai.readthedocs.io/en/latest/gallery/tutorials/bias/mitigating_bias/binary_classification/examples/example_census_data.html)
- [HolisticAI Reweighing API](https://holisticai.readthedocs.io/en/latest/reference/bias/.generated/holisticai.bias.mitigation.Reweighing.html)
- [Fairlearn fairness metrics](https://fairlearn.org/main/user_guide/assessment/common_fairness_metrics.html)
- [AIF360 Reweighing](https://aif360.readthedocs.io/en/latest/modules/generated/aif360.sklearn.preprocessing.Reweighing.html)